# 02b — CSV Data Exploration (full)

**From Clinical Case Reports to Knowledge Graphs**

Explores the full CSV export produced by
[`01_data_preparation.ipynb`](01_data_preparation.ipynb) in
`data/csv/full/` — every case and every article, no sampling. DuckDB queries
the CSV files directly through `read_csv_auto` — no separate database file
is involved — so the same SQL used here works unchanged against the
50-article sample in
[`02a_csv_data_exploraton_sample.ipynb`](02a_csv_data_exploraton_sample.ipynb)
and the DuckDB database in
[`03_duckdb_data_exploraton.ipynb`](03_duckdb_data_exploraton.ipynb).

Because array-typed columns (`authors`, `mesh_terms`, `major_mesh_terms`,
`keywords`) round-trip through CSV as bracketed text rather than native
arrays, `DESCRIBE` below shows them as `VARCHAR` — see Section 5 of
`01_data_preparation.ipynb` for why. Queries over the full CSVs are
noticeably slower than over the sample or the DuckDB database, since DuckDB
re-parses the CSV text on every query instead of reading a pre-typed
columnar table.


## 1. Setup


In [ ]:
from pathlib import Path

import duckdb

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent
CSV_DIR = PROJECT_DIR / "data" / "csv" / "full"

if not (CSV_DIR / "cases.csv").exists():
    raise FileNotFoundError(
        f"{CSV_DIR} not found — run 01_data_preparation.ipynb first."
    )

con = duckdb.connect()  # in-memory
for table in ("cases", "metadata", "data_dictionary"):
    con.execute(
        f"CREATE VIEW {table} AS SELECT * FROM read_csv_auto('{(CSV_DIR / f'{table}.csv').as_posix()}')"
    )

con.sql("SHOW TABLES")


## 2. Example queries — exploring the full data


## Table schema


In [ ]:
con.sql("DESCRIBE cases")


In [ ]:
con.sql("DESCRIBE metadata")


### A random sample of cases


In [ ]:
con.sql("SELECT * FROM cases LIMIT 5")


In [ ]:
con.sql("SELECT * FROM metadata LIMIT 5")


### Text length distribution


In [ ]:
con.sql("""
    SELECT
        MIN(LENGTH(case_text)) AS min_chars,
        MEDIAN(LENGTH(case_text)) AS median_chars,
        AVG(LENGTH(case_text))::INT AS avg_chars,
        MAX(LENGTH(case_text)) AS max_chars
    FROM cases
""")


### Patient demographics


In [ ]:
con.sql("""
    SELECT gender, COUNT(*) AS n_cases
    FROM cases
    GROUP BY gender
    ORDER BY n_cases DESC
""")


In [ ]:
con.sql("""
    SELECT
        MIN(age) AS min_age,
        MEDIAN(age) AS median_age,
        AVG(age)::INT AS avg_age,
        MAX(age) AS max_age
    FROM cases
    WHERE age IS NOT NULL
""")


### Joining `cases` with `metadata`

`cases` and `metadata` share the `article_id` (PMCID) column, so we can
bring in publication year, journal, license, etc.


In [ ]:
con.sql("""
    SELECT m.year, COUNT(*) AS n_cases
    FROM cases AS c
    JOIN metadata AS m USING (article_id)
    GROUP BY m.year
    ORDER BY m.year
""")


In [ ]:
con.sql("""
    SELECT journal, COUNT(DISTINCT article_id) AS n_articles
    FROM metadata
    GROUP BY journal
    ORDER BY n_articles DESC
    LIMIT 10
""")


In [ ]:
con.sql("""
    SELECT license, COUNT(*) AS n_articles
    FROM metadata
    GROUP BY license
    ORDER BY n_articles DESC
""")


## Wrap up


In [ ]:
con.close()
